# 🐾 Detección de Puntos Clave con Aprendizaje por Transferencia

**Autor original:** [Sayak Paul](https://twitter.com/RisingSayak), convertido a Keras 3 por [Muhammad Anas Raza](https://anasrz.com)  
**Fecha original:** 02/05/2021 | **Última modificación original:** 19/07/2023  
**Adaptación al español y material pedagógico:** Alfredo Díaz Claro, 2026  

---

## 🎯 ¿Qué aprenderás en este cuaderno?

Este cuaderno es un **material de autoestudio** que te guiará paso a paso para construir un detector de puntos clave (*keypoints*) usando técnicas modernas de visión por computador y aprendizaje profundo. Al finalizar, habrás aprendido a:

- Comprender qué es la detección de puntos clave y sus aplicaciones.
- Cargar y preprocesar un conjunto de datos con anotaciones de puntos clave.
- Aplicar **aumento de datos** (data augmentation) respetando las coordenadas de los puntos.
- Usar **aprendizaje por transferencia** con MobileNetV2 como red base.
- Construir, entrenar y evaluar un modelo de regresión de coordenadas.

---

## 📚 Conceptos previos recomendados

Para aprovechar al máximo este cuaderno, se recomienda tener conocimientos básicos de:
- Redes neuronales convolucionales (CNN).
- Uso básico de Python, NumPy y Matplotlib.
- Fundamentos de Keras/TensorFlow.

---

## 🔍 Descripción del problema

La **detección de puntos clave** consiste en localizar partes específicas de un objeto dentro de una imagen. Por ejemplo, en un rostro humano los puntos clave pueden ser la punta de la nariz, las comisuras de los ojos, las cejas, etc. En este cuaderno trabajaremos con **imágenes de perros**, detectando 24 puntos anatómicos clave por imagen.

Esta técnica tiene aplicaciones en:
- 🏃 Estimación de pose humana o animal.
- 😊 Reconocimiento y análisis facial.
- 🎮 Animación de personajes en videojuegos.
- 🏥 Análisis biomecánico en medicina.

> **Nota técnica:** Este cuaderno requiere TensorFlow 2.4 o superior y la librería [`imgaug`](https://imgaug.readthedocs.io/), que instalaremos a continuación.

## 🛠️ Paso 1 — Instalación de dependencias

Antes de comenzar, necesitamos instalar la librería `imgaug`, especializada en **aumento de datos para imágenes con anotaciones** (puntos clave, bounding boxes, segmentaciones). Esta librería es fundamental porque, al transformar una imagen geométricamente (rotar, escalar, voltear), también transforma automáticamente las coordenadas de los puntos clave.

> ⚠️ **¿Por qué es importante `imgaug`?**  
> Si aplicamos una rotación de 10° a una imagen pero no ajustamos las coordenadas de los puntos clave, el modelo aprenderá posiciones incorrectas. `imgaug` resuelve esto de forma transparente.

In [1]:
pip install --upgrade git+https://github.com/aleju/imgaug.git

  Cloning https://github.com/aleju/imgaug.git to /tmp/pip-req-build-za7rm0wm
  Running command git clone --filter=blob:none --quiet https://github.com/aleju/imgaug.git /tmp/pip-req-build-za7rm0wm
  Resolved https://github.com/aleju/imgaug.git to commit 0101108d4fed06bc5056c4a03e2bcb0216dac326
  Preparing metadata (setup.py) ... done


---

## 📚 Importación de librerías

A continuación importamos todas las librerías necesarias. A modo de repaso:

| Librería | Uso en este cuaderno |
|---|---|
| `keras` / `layers` | Construcción y entrenamiento del modelo de red neuronal |
| `imgaug` | Aumento de datos con soporte para puntos clave |
| `PIL` (Pillow) | Manejo de imágenes (lectura, conversión de formatos) |
| `sklearn` | División del dataset en entrenamiento y validación |
| `matplotlib` | Visualización de imágenes y puntos clave |
| `pandas` | Lectura del archivo CSV con metadatos de puntos clave |
| `numpy` | Operaciones matriciales y manejo de arrays |
| `json`, `os` | Lectura de archivos de anotaciones y rutas del sistema |

---

## 📦 Paso 2 — Descarga del conjunto de datos

### El dataset StanfordExtra

Utilizaremos el [**dataset StanfordExtra**](https://github.com/benjiebob/StanfordExtra), que contiene:
- 📸 **12,000 imágenes** de perros de distintas razas.
- 📍 **Anotaciones de 24 puntos clave** por imagen (articulaciones y partes corporales).
- 🗺️ Máscaras de segmentación.

Este dataset fue construido a partir del [Stanford Dogs Dataset](http://vision.stanford.edu/aditya86/ImageNetDogs/), que a su vez es un subconjunto de ImageNet.

### Descarga de imágenes

Las imágenes se descargan públicamente desde el servidor de Stanford:

In [2]:
from keras import layers
import keras
import numpy as np

# Definir sctypes manualmente para engañar a librerías antiguas
if not hasattr(np, 'sctypes'):
    np.sctypes = {'int': [np.int8, np.int16, np.int32, np.int64],
                  'uint': [np.uint8, np.uint16, np.uint32, np.uint64],
                  'float': [np.float16, np.float32, np.float64],
                  'complex': [np.complex64, np.complex128],
                  'others': [bool, object, str, bytes]}

from imgaug.augmentables.kps import KeypointsOnImage, Keypoint
# OR more simply
from imgaug.augmentables import Keypoint, KeypointsOnImage
import imgaug.augmenters as iaa


from PIL import Image
from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt
import pandas as pd
import json
import os

In [3]:
!wget -q http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar

### Descarga de anotaciones

> ⚠️ **Requisito adicional:** Las anotaciones JSON del dataset StanfordExtra se distribuyen por solicitud expresa de los autores. Para obtenerlas, debes completar [este formulario](https://forms.gle/sRtbicgxsWvRtRmUA). Los autores piden explícitamente que no se comparta el archivo JSON, y este cuaderno respeta esa condición.

Revisa tu correo (el que puso en el formulario) y descarga las anotaciones.

Una vez descargadas, coloca el archivo `stanfordextra_v12.zip` en tu directorio de trabajo (el directorio raíz `~/` en Colab).

Si deseas cargar los archivos desde tu drive, monta la unidad, si no, no es necesario ejecutar el siguiente paso

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Extracción de los archivos

Ubicara la carpeta donde cargo el zip y ejecuta la siguiente celda para descomprimir los archivos:

In [9]:
!wget -q "https://drive.google.com/uc?export=download&id=1v5w2lMfjE3FZRPXsJKUli7NaKhVFvBGj" -O archivo.zip


In [8]:
!tar xf images.tar
!unzip -qq /content/StanfordExtra_v12.zip

unzip:  cannot find or open /content/StanfordExtra_v12.zip, /content/StanfordExtra_v12.zip.zip or /content/StanfordExtra_v12.zip.ZIP.


In [ ]:
import numpy as np
import json
import os
import matplotlib.pyplot as plt
from PIL import Image

# --- Definiciones copiadas de celdas anteriores para autocontención ---
IMG_DIR = "Images"
JSON = "StanfordExtra_V12/StanfordExtra_v12.json"

# Cargar las anotaciones del archivo JSON.
with open(JSON) as infile:
    json_data = json.load(infile)

# Construir un diccionario que mapea la ruta de la imagen con su información.
json_dict = {i["img_path"]: i for i in json_data}

# Función para leer una imagen y obtener sus anotaciones. (Copiado de la celda 14)
def get_dog(name):
    data = json_dict[name]
    img_data = plt.imread(os.path.join(IMG_DIR, data["img_path"]))
    # Si la imagen es RGBA, convertirla a RGB.
    if img_data.shape[-1] == 4:
        img_data = img_data.astype(np.uint8)
        img_data = Image.fromarray(img_data)
        img_data = np.array(img_data.convert("RGB"))
    data["img_data"] = img_data

    return data

# Definir visualize_images_only_grid (función que se había prometido generar previamente)
def visualize_images_only_grid(images, titles, rows, cols):
    fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(16, 12))
    axes = axes.flatten() # Aplanar para facilitar la iteración

    for i, (ax, image, title) in enumerate(zip(axes, images, titles)):
        ax.imshow(image)
        ax.set_title(title, fontsize=10)
        ax.axis("off")

    plt.tight_layout(pad=2.0)
    plt.show()
# --- Fin de las definiciones copiadas ---

num_samples = 20
samples = list(json_dict.keys())
selected_samples = np.random.choice(samples, num_samples, replace=False)

images_grid, titles_grid = [], []

for sample_key in selected_samples:
    data = get_dog(sample_key)
    images_grid.append(data["img_data"])

    # Extraer la raza del perro de la ruta de la imagen
    path_parts = data["img_path"].split('/')
    breed_name = path_parts[0].split('-', 1)[1].replace('_', ' ').title() if '-' in path_parts[0] else path_parts[0]
    titles_grid.append(breed_name)

print(f"Visualizando {num_samples} muestras aleatorias con nombre de raza:")
visualize_images_only_grid(images_grid, titles=titles_grid, rows=5, cols=4)

---

## ⚙️ Paso 3 — Definición de hiperparámetros

Los **hiperparámetros** son configuraciones del modelo y del proceso de entrenamiento que se definen antes de comenzar y no se aprenden durante el entrenamiento. Aquí definimos cuatro:

| Hiperparámetro | Valor | Descripción |
|---|---|---|
| `IMG_SIZE` | 224 | Tamaño (en píxeles) al que se redimensionarán todas las imágenes. Se usa 224×224 porque es el tamaño estándar de entrada para MobileNetV2. |
| `BATCH_SIZE` | 64 | Número de imágenes procesadas simultáneamente en cada paso de entrenamiento. Valores mayores aceleran el entrenamiento pero consumen más memoria. |
| `EPOCHS` | 5 | Número de veces que el modelo verá el dataset completo durante el entrenamiento. |
| `NUM_KEYPOINTS` | 48 | Total de valores a predecir: 24 puntos × 2 coordenadas (x, y) = 48 valores. |

> 💡 **Para experimentar:** Prueba aumentar `EPOCHS` a 20 o reducir `BATCH_SIZE` a 32 y observa cómo afecta la velocidad y precisión del entrenamiento.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 5
NUM_KEYPOINTS = 24 * 2  # 24 puntos, cada uno con coordenadas x e y

---

## 📂 Paso 5 — Carga de datos

### Estructura de las anotaciones

El archivo JSON del dataset contiene una lista de entradas, donde cada entrada corresponde a una imagen. Para facilitar la búsqueda, construiremos un **diccionario** (`json_dict`) que mapea la ruta de cada imagen con sus anotaciones.

Además, cargaremos el archivo CSV de definiciones de puntos clave, que nos proporciona:
- Los **nombres** de cada punto clave (ej: "hombro izquierdo", "pata delantera").
- Los **colores** para visualizarlos en las gráficas.

Finalmente, definimos la función `get_dog()`, que dado el nombre de una imagen, retorna tanto la imagen como sus coordenadas de puntos clave. Esta función también convierte imágenes RGBA a RGB, ya que el modelo trabaja con 3 canales de color.

In [ ]:
IMG_DIR = "Images"
JSON = "StanfordExtra_V12/StanfordExtra_v12.json"
KEYPOINT_DEF = (
    "https://github.com/benjiebob/StanfordExtra/raw/master/keypoint_definitions.csv"
)

# Cargar las anotaciones del archivo JSON.
with open(JSON) as infile:
    json_data = json.load(infile)

# Construir un diccionario que mapea la ruta de la imagen con su información.
json_dict = {i["img_path"]: i for i in json_data}

### Estructura de una entrada del JSON

Una entrada típica de `json_dict` tiene la siguiente forma:

```python
'n02085782-Japanese_spaniel/n02085782_2886.jpg': {
  'img_bbox': [205, 20, 116, 201],   # Bounding box del perro
  'img_height': 272,
  'img_path': 'n02085782-Japanese_spaniel/n02085782_2886.jpg',
  'img_width': 350,
  'is_multiple_dogs': False,
  'joints': [
    [108.67, 252.0, 1],   # [x, y, visibilidad]
    [147.67, 229.0, 1],
    ...
    [0, 0, 0],            # Punto no anotado
  ],
  'seg': ...              # Máscara de segmentación
}
```

### Sobre el campo `joints`

El campo `joints` contiene exactamente **24 entradas**, una por cada punto clave del cuerpo del perro. Cada entrada es una lista `[x, y, visibilidad]`:

- **x, y**: coordenadas del punto en píxeles dentro de la imagen original.
- **visibilidad**: `1` = punto visible y anotado, `0` = punto no anotado o no visible.

> 💡 **Decisión de diseño:** En este ejemplo, tratamos tanto los puntos no visibles como los no anotados de la misma manera (como coordenadas en el origen), lo que permite hacer entrenamiento con mini-lotes (*mini-batch learning*) sin complejidad adicional.

In [ ]:
# Cargar el archivo de definición de puntos clave y previsualizarlo.
keypoint_def = pd.read_csv(KEYPOINT_DEF)
keypoint_def.head()

# Extraer los colores y etiquetas.
colours = keypoint_def["Hex colour"].values.tolist()
colours = ["#" + colour for colour in colours]
labels = keypoint_def["Name"].values.tolist()


# Función para leer una imagen y obtener sus anotaciones.
def get_dog(name):
    data = json_dict[name]
    img_data = plt.imread(os.path.join(IMG_DIR, data["img_path"]))
    # Si la imagen es RGBA, convertirla a RGB.
    if img_data.shape[-1] == 4:
        img_data = img_data.astype(np.uint8)
        img_data = Image.fromarray(img_data)
        img_data = np.array(img_data.convert("RGB"))
    data["img_data"] = img_data

    return data

---

## 👀 Paso 6 — Visualización de los datos

Antes de entrenar cualquier modelo, es fundamental **explorar visualmente** los datos. Esto nos ayuda a:
- Verificar que las anotaciones son correctas.
- Entender la variedad y distribución de las imágenes.
- Detectar posibles errores en los datos.

La función `visualize_keypoints()` recibe una lista de imágenes y sus puntos clave, y muestra para cada imagen:
- **Columna izquierda:** La imagen original sin anotaciones.
- **Columna derecha:** La imagen con los 24 puntos clave superpuestos, cada uno con un color único.

> 🔍 **Observa:** Las imágenes tienen tamaños variables (algunas son rectangulares, otras cuadradas). Esto es lo habitual en datasets del mundo real y es un desafío que debemos manejar al preprocesar los datos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from imgaug.augmentables.kps import KeypointsOnImage, Keypoint # Import the missing Keypoint class

# Parte de este código proviene de:
# https://github.com/benjiebob/StanfordExtra/blob/master/demo.ipynb
def visualize_keypoints(images, keypoints):
    # Ensure keypoints is a list for consistency
    if not isinstance(keypoints, list):
        keypoints = [keypoints]

    fig, axes = plt.subplots(nrows=len(images), ncols=2, figsize=(16, 12))

    # Ensure axes is always a 2D array of shape (N, 2) even if N=1
    if len(images) == 1 and axes.ndim == 1:
        axes = np.expand_dims(axes, axis=0)

    [ax.axis("off") for ax in np.ravel(axes)]

    for (ax_orig, ax_all), image, current_keypoint in zip(axes, images, keypoints):
        ax_orig.imshow(image)
        ax_all.imshow(image)

        # Si los puntos fueron generados por `imgaug`, las coordenadas
        # se iteran de forma diferente.
        if isinstance(current_keypoint, KeypointsOnImage):
            for idx, kp in enumerate(current_keypoint.keypoints):
                ax_all.scatter(
                    [kp.x],
                    [kp.y],
                    c=colours[idx],
                    marker="x",
                    s=50,
                    linewidths=5,
                )
        else:
            current_keypoint = np.array(current_keypoint)
            # Descartamos la bandera de visibilidad (tercer valor).
            current_keypoint = current_keypoint[:, :2]
            for idx, (x, y) in enumerate(current_keypoint):
                ax_all.scatter([x], [y], c=colours[idx], marker="x", s=50, linewidths=5)

    plt.tight_layout(pad=2.0)
    plt.show()


# Seleccionar cuatro muestras aleatorias para visualizar.
samples = list(json_dict.keys())
num_samples = 4
selected_samples = np.random.choice(samples, num_samples, replace=False)

images, keypoints = [], []

for sample in selected_samples:
    data = get_dog(sample)
    image = data["img_data"]
    keypoint = data["joints"]

    images.append(image)
    keypoints.append(keypoint)

visualize_keypoints(images, keypoints)

### Reflexión sobre los datos visualizados

Como puedes observar, las imágenes tienen **tamaños no uniformes**, lo cual es muy común en escenarios del mundo real. Si redimensionamos estas imágenes directamente a 224×224, las coordenadas de los puntos clave también cambiarán proporcionalmente.

Además, si aplicamos transformaciones geométricas como **volteo horizontal** o **rotación**, los puntos clave deben transformarse de la misma manera para mantener la coherencia.

Afortunadamente, `imgaug` proporciona herramientas que manejan todo esto automáticamente. En la siguiente sección construiremos un **generador de datos** que aplica estas transformaciones de manera consistente.

---

## 🔄 Paso 7 — Generador de datos con aumento

### ¿Qué es un generador de datos?

En lugar de cargar todas las imágenes en memoria (lo cual podría no ser posible con datasets grandes), usamos un **generador de datos**. Este carga y procesa las imágenes en lotes (*batches*) bajo demanda durante el entrenamiento.

Nuestra clase `KeyPointsDataset` hereda de `keras.utils.PyDataset`, lo que nos permite:
- Procesamiento en lotes eficiente.
- Soporte para carga en múltiples hilos (`workers`).
- Barajar los datos al final de cada época.

### Normalización de coordenadas

Un detalle importante: al final del método `__data_generation`, las coordenadas se **dividen entre `IMG_SIZE`** para normalizarlas al rango `[0, 1]`. Esto es una buena práctica en problemas de regresión porque:
- Hace que los valores objetivo sean comparables entre sí.
- Mejora la estabilidad numérica del entrenamiento.
- Es más fácil para el modelo aprender a predecir valores en `[0, 1]` que coordenadas en píxeles absolutas.

### Forma de salida del generador

Los puntos clave se almacenan con forma `(batch_size, 1, 1, 48)`. Esta forma particular es necesaria para que coincida con la salida de nuestra red neuronal completamente convolucional. Lo veremos en detalle en la sección de construcción del modelo.

In [ ]:
class KeyPointsDataset(keras.utils.PyDataset):
    def __init__(self, image_keys, aug, batch_size=BATCH_SIZE, train=True, **kwargs):
        super().__init__(**kwargs)
        self.image_keys = image_keys
        self.aug = aug
        self.batch_size = batch_size
        self.train = train
        self.on_epoch_end()

    def __len__(self):
        return len(self.image_keys) // self.batch_size

    def on_epoch_end(self):
        self.indexes = np.arange(len(self.image_keys))
        if self.train:
            np.random.shuffle(self.indexes)

    def __getitem__(self, index):
        indexes = self.indexes[index * self.batch_size : (index + 1) * self.batch_size]
        image_keys_temp = [self.image_keys[k] for k in indexes]
        images, keypoints = self.__data_generation(image_keys_temp)

        return (images, keypoints)

    def __data_generation(self, image_keys_temp):
        batch_images = np.empty((self.batch_size, IMG_SIZE, IMG_SIZE, 3), dtype="int")
        batch_keypoints = np.empty(
            (self.batch_size, 1, 1, NUM_KEYPOINTS), dtype="float32"
        )

        for i, key in enumerate(image_keys_temp):
            data = get_dog(key)
            current_keypoint = np.array(data["joints"])[:, :2]
            kps = []

            # Crear objetos Keypoint con las coordenadas originales
            # para poder aplicar el pipeline de aumento de datos.
            for j in range(0, len(current_keypoint)):
                kps.append(Keypoint(x=current_keypoint[j][0], y=current_keypoint[j][1]))

            # Proyectar la imagen original y sus coordenadas de puntos clave.
            current_image = data["img_data"]
            kps_obj = KeypointsOnImage(kps, shape=current_image.shape)

            # Aplicar el pipeline de aumento de datos.
            new_image, new_kps_obj = self.aug(image=current_image, keypoints=kps_obj)
            batch_images[i,] = new_image

            # Extraer las coordenadas del objeto de puntos clave transformado.
            kp_temp = []
            for keypoint in new_kps_obj:
                kp_temp.append(np.nan_to_num(keypoint.x))
                kp_temp.append(np.nan_to_num(keypoint.y))

            # Remodelar para coincidir con la forma de salida del modelo.
            batch_keypoints[i,] = np.array(kp_temp).reshape(1, 1, 24 * 2)

        # Normalizar las coordenadas al rango [0, 1].
        batch_keypoints = batch_keypoints / IMG_SIZE

        return (batch_images, batch_keypoints)

> 📖 **Para profundizar:** Si quieres aprender más sobre cómo trabajar con puntos clave en `imgaug`, consulta la [documentación oficial](https://imgaug.readthedocs.io/en/latest/source/examples_keypoints.html).

---

## 🎨 Paso 8 — Definición de transformaciones de aumento

### ¿Qué es el aumento de datos?

El **aumento de datos** (*data augmentation*) es una técnica para aumentar artificialmente la diversidad del dataset de entrenamiento aplicando transformaciones aleatorias a las imágenes. Esto ayuda a que el modelo:
- Sea más robusto frente a variaciones en la posición, escala u orientación.
- No memorice el conjunto de entrenamiento (*overfitting*).
- Generalice mejor a imágenes nuevas.

### Transformaciones aplicadas durante el entrenamiento (`train_aug`)

| Transformación | Descripción |
|---|---|
| `iaa.Resize(224)` | Redimensiona todas las imágenes a 224×224 (obligatorio para la red). |
| `iaa.Fliplr(0.3)` | Voltea horizontalmente el 30% de las imágenes. |
| `iaa.Affine(rotate=10, scale=(0.5, 0.7))` | Aplica aleatoriamente rotación (±10°) y zoom (50–70%) al 30% de las imágenes. |

### Transformaciones de validación/prueba (`test_aug`)

Durante la evaluación **no aplicamos transformaciones aleatorias**, ya que queremos medir el rendimiento real del modelo. Solo redimensionamos las imágenes al tamaño requerido.

In [ ]:
train_aug = iaa.Sequential(
    [
        iaa.Resize(IMG_SIZE, interpolation="linear"),
        iaa.Fliplr(0.3),
        # `Sometimes()` aplica una transformación aleatoriamente con
        # una probabilidad dada (0.3 en este caso).
        iaa.Sometimes(0.3, iaa.Affine(rotate=10, scale=(0.5, 0.7))),
    ]
)

test_aug = iaa.Sequential([iaa.Resize(IMG_SIZE, interpolation="linear")])

---

## ✂️ Paso 9 — División del dataset

Dividimos el conjunto de datos en dos subconjuntos:

- **Entrenamiento (85%):** El modelo aprende de estas imágenes durante el entrenamiento.
- **Validación (15%):** Estas imágenes se usan para evaluar el modelo al final de cada época, **sin** que el modelo aprenda de ellas. Esto nos permite detectar *overfitting*.

> 💡 **Buena práctica:** Siempre barajar los datos antes de dividirlos para evitar sesgos por el orden en que fueron recolectados.

In [ ]:
np.random.shuffle(samples)
train_keys, validation_keys = (
    samples[int(len(samples) * 0.15) :],
    samples[: int(len(samples) * 0.15)],
)

---

## 🔬 Paso 10 — Verificación del generador de datos

Antes de entrenar el modelo, es una buena práctica **inspeccionar las salidas del generador** para asegurarnos de que:
1. Las imágenes se cargan correctamente.
2. Las coordenadas de los puntos clave están normalizadas en `[0, 1]`.
3. Las transformaciones de aumento se ven razonables visualmente.

Ejecuta esta celda y examina las imágenes generadas para verificar que los puntos clave se transformaron correctamente junto con las imágenes.

In [ ]:
train_dataset = KeyPointsDataset(
    train_keys, train_aug, workers=2, use_multiprocessing=True
)
validation_dataset = KeyPointsDataset(
    validation_keys, test_aug, train=False, workers=2, use_multiprocessing=True
)

print(f"Total de lotes en conjunto de entrenamiento: {len(train_dataset)}")
print(f"Total de lotes en conjunto de validación: {len(validation_dataset)}")

# Verificar que las coordenadas están correctamente normalizadas
sample_images, sample_keypoints = next(iter(train_dataset))
assert sample_keypoints.max() == 1.0
assert sample_keypoints.min() == 0.0

# Visualizar 4 muestras del generador de entrenamiento
sample_keypoints = sample_keypoints[:4].reshape(-1, 24, 2) * IMG_SIZE
visualize_keypoints(sample_images[:4], sample_keypoints)

---

## 🏗️ Paso 11 — Construcción del modelo

### Aprendizaje por transferencia

El **aprendizaje por transferencia** (*transfer learning*) consiste en reutilizar un modelo preentrenado en una tarea como punto de partida para una tarea diferente. En lugar de entrenar una red desde cero (lo que requeriría millones de imágenes y días de cómputo), tomamos las características que una red ya aprendió y las adaptamos a nuestro problema.

### ¿Por qué MobileNetV2?

Usamos **MobileNetV2** preentrenado en **ImageNet-1k** como red base por varias razones:

1. **Conocimiento previo relevante:** El Stanford Dogs Dataset fue construido a partir de ImageNet, por lo que MobileNetV2 ya "conoce" perros de muchas razas y ha aprendido a detectar sus características visuales.
2. **Eficiencia:** MobileNetV2 fue diseñado para ser liviano y eficiente, ideal para experimentos rápidos.
3. **Disponibilidad:** Está incluido directamente en Keras.

### Arquitectura del modelo

```
Entrada (224×224×3)
    ↓
Preprocesamiento MobileNetV2
    ↓
MobileNetV2 (congelado, pesos ImageNet)
    ↓
Dropout (0.3) — regularización
    ↓
SeparableConv2D(48 filtros, kernel 5×5, ReLU)
    ↓
SeparableConv2D(48 filtros, kernel 3×3, Sigmoid) → Salida (1×1×48)
```

### Red completamente convolucional

La red es **completamente convolucional** (no tiene capas densas/fully-connected al final). Esto tiene ventajas:
- Menos parámetros que una versión con capas densas equivalentes.
- La salida `(None, 1, 1, 48)` mapea directamente a nuestros 48 valores de coordenadas normalizadas.
- La función de activación **Sigmoid** en la última capa garantiza que las salidas estén en `[0, 1]`, consistente con nuestra normalización de coordenadas.

In [ ]:
def get_model():
    # Cargar los pesos preentrenados de MobileNetV2 y congelarlos.
    backbone = keras.applications.MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    backbone.trainable = False

    inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3))
    x = keras.applications.mobilenet_v2.preprocess_input(inputs)
    x = backbone(x)
    x = layers.Dropout(0.3)(x)
    x = layers.SeparableConv2D(
        NUM_KEYPOINTS, kernel_size=5, strides=1, activation="relu"
    )(x)
    outputs = layers.SeparableConv2D(
        NUM_KEYPOINTS, kernel_size=3, strides=1, activation="sigmoid"
    )(x)

    return keras.Model(inputs, outputs, name="detector_puntos_clave")

### Inspección del resumen del modelo

Ejecuta la siguiente celda para ver la arquitectura completa. Presta atención a:
- El número total de parámetros del modelo.
- Cuántos parámetros son **entrenables** (nuestras capas de regresión) vs. **no entrenables** (backbone congelado).
- La forma de la salida: debería ser `(None, 1, 1, 48)`.

In [ ]:
get_model().summary()

### ¿Por qué la forma de salida es `(None, 1, 1, 48)`?

La forma `(None, 1, 1, 48)` significa:
- **None:** Tamaño de lote (variable).
- **1, 1:** Dimensiones espaciales reducidas a 1×1 por las capas convolucionales.
- **48:** Los 48 valores de salida (24 puntos × 2 coordenadas).

Por eso en el generador de datos almacenamos los puntos clave con `reshape(1, 1, 24 * 2)`: para que coincidan con esta forma de salida al calcular la pérdida durante el entrenamiento.

---

## 🚀 Paso 12 — Compilación y entrenamiento del modelo

### Función de pérdida: Error Cuadrático Medio (MSE)

Usamos **MSE** (*Mean Squared Error*) como función de pérdida porque este es un problema de **regresión** (predecir valores continuos, no clases). El MSE penaliza fuertemente las predicciones muy alejadas del valor real, lo que motiva al modelo a hacer predicciones precisas.

$$MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

### Optimizador: Adam

**Adam** es uno de los optimizadores más utilizados en deep learning. Combina las ventajas de los optimizadores *Momentum* y *RMSprop*, adaptando la tasa de aprendizaje para cada parámetro individualmente. Usamos una tasa de aprendizaje de `1e-4` (0.0001).

> ⏱️ **Tiempo de entrenamiento:** Con solo 5 épocas, el entrenamiento será rápido pero el modelo no alcanzará su máximo rendimiento. Esto es suficiente para validar que la arquitectura funciona correctamente.

In [ ]:
model = get_model()
model.compile(loss="mse", optimizer=keras.optimizers.Adam(1e-4))
model.fit(train_dataset, validation_data=validation_dataset, epochs=EPOCHS)

---

## 📊 Paso 13 — Predicciones y visualización de resultados

Una vez entrenado el modelo, lo evaluamos visualmente comparando:
- Las anotaciones **reales** (ground truth) del conjunto de validación.
- Las **predicciones** del modelo sobre las mismas imágenes.

Esta comparación visual nos da una intuición inmediata de cuán bien está funcionando el modelo. Un buen indicador es que los puntos predichos estén cerca de los reales en las partes visibles del cuerpo del perro.

> 💡 **Recuerda:** Las predicciones del modelo están en el rango `[0, 1]` (normalizadas), por lo que debemos multiplicarlas por `IMG_SIZE` para obtener las coordenadas en píxeles.

In [ ]:
sample_val_images, sample_val_keypoints = next(iter(validation_dataset))
sample_val_images = sample_val_images[:4]
sample_val_keypoints = sample_val_keypoints[:4].reshape(-1, 24, 2) * IMG_SIZE
predictions = model.predict(sample_val_images).reshape(-1, 24, 2) * IMG_SIZE

# Visualizar anotaciones reales
print("📍 Puntos clave reales (Ground Truth):")
visualize_keypoints(sample_val_images, sample_val_keypoints)

# Visualizar predicciones del modelo
print("🤖 Predicciones del modelo:")
visualize_keypoints(sample_val_images, predictions)

### Interpretación de los resultados

Con solo 5 épocas de entrenamiento, es probable que las predicciones no sean perfectas, pero deberían mostrar una tendencia correcta: los puntos predichos deberían estar aproximadamente en las zonas correctas del cuerpo del perro.

Entrenar por más épocas mejorará progresivamente los resultados.

---

## 🚀 Para seguir aprendiendo

Una vez que hayas completado este cuaderno, te proponemos los siguientes desafíos para profundizar:

### 🔧 Experimentos de aumento de datos
Prueba diferentes transformaciones de `imgaug` en `train_aug` e investiga cómo afectan el rendimiento:
- `iaa.GaussianBlur()` — desenfoque gaussiano.
- `iaa.AdditiveGaussianNoise()` — ruido aleatorio.
- `iaa.Crop(percent=(0, 0.1))` — recorte aleatorio.

### 🎯 Fine-tuning del backbone
En este cuaderno **congelamos** el backbone MobileNetV2 (no actualizamos sus pesos). Prueba **descongelar** las últimas capas y hacer *fine-tuning*:
```python
backbone.trainable = True
# Opcionalmente, congelar solo las primeras capas:
for layer in backbone.layers[:-20]:
    layer.trainable = False
```
Consulta la [guía de transfer learning de Keras](https://keras.io/guides/transfer_learning/) para más detalles.

### 🏛️ Otras arquitecturas
Prueba reemplazar MobileNetV2 por otras arquitecturas como:
- `keras.applications.EfficientNetB0` — más preciso pero más pesado.
- `keras.applications.ResNet50` — clásico y robusto.
- `keras.applications.MobileNetV3Small` — más eficiente que V2.

### ⏳ Más épocas de entrenamiento
Entrena el modelo durante 20, 50 o 100 épocas y monitorea cómo evoluciona la pérdida de validación. Agrega callbacks como `EarlyStopping` y `ReduceLROnPlateau` para un entrenamiento más inteligente.

---

## 📖 Recursos adicionales

- 🐶 **Dataset StanfordExtra:** [GitHub](https://github.com/benjiebob/StanfordExtra)
- 📐 **Documentación imgaug:** [imgaug.readthedocs.io](https://imgaug.readthedocs.io/)
- 🧠 **Guía de Transfer Learning en Keras:** [keras.io](https://keras.io/guides/transfer_learning/)
- 📊 **MobileNetV2 - Artículo original:** [arxiv.org](https://arxiv.org/abs/1801.04381)

---

*Adaptación al español realizada por **Alfredo Díaz Claro**, 2026.*  
*Material de autoestudio — Visión por Computador con Deep Learning.*